# Project 5: Quantum Portfolio Optimization

## Overview

Portfolio optimization is a cornerstone of quantitative finance. The **Markowitz mean-variance model** (1952) seeks the optimal allocation of assets to maximize expected return for a given level of risk. This combinatorial optimization problem maps naturally onto quantum computing frameworks.

### The Markowitz Model

Given $n$ assets with expected returns $\boldsymbol{\mu} \in \mathbb{R}^n$ and covariance matrix $\Sigma \in \mathbb{R}^{n \times n}$, the classical mean-variance optimization problem is:

$$\min_{\mathbf{w}} \quad \frac{1}{2} \mathbf{w}^T \Sigma \mathbf{w} - q \, \boldsymbol{\mu}^T \mathbf{w}$$

subject to $\sum_i w_i = 1$, $w_i \geq 0$, where $q$ is a risk aversion parameter.

### QAOA Approach

The **Quantum Approximate Optimization Algorithm (QAOA)** is a hybrid quantum-classical algorithm designed for combinatorial optimization:

1. Encode the optimization problem as an **Ising Hamiltonian** $H_C$
2. Prepare a quantum state using alternating layers of **cost** ($e^{-i\gamma H_C}$) and **mixer** ($e^{-i\beta H_M}$) unitaries
3. Measure the expectation value $\langle H_C \rangle$ and optimize the variational parameters $(\gamma, \beta)$ classically
4. Sample the optimal quantum state to extract the solution

For portfolio selection with binary decisions (include/exclude each asset), QAOA provides a natural framework that can explore exponentially many portfolios simultaneously through quantum superposition.

In [ ]:
# --- Imports ---
import pennylane as qml
from pennylane import numpy as pnp
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import FancyBboxPatch

from scipy.optimize import minimize, minimize_scalar
from itertools import combinations

import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

print(f"PennyLane version: {qml.__version__}")
print(f"NumPy version    : {np.__version__}")

## Theory: From Mean-Variance to Quantum Optimization

### Mean-Variance Optimization

The portfolio optimization problem seeks weights $\mathbf{w}$ that minimize risk (variance) while achieving target return:

$$\min_{\mathbf{w}} \quad \frac{1}{2} \mathbf{w}^T \Sigma \mathbf{w} - q \, \boldsymbol{\mu}^T \mathbf{w}$$

where:
- $\mathbf{w} \in \{0, 1\}^n$ is the binary selection vector (invest or not)
- $\Sigma$ is the asset covariance matrix
- $\boldsymbol{\mu}$ is the expected return vector
- $q$ controls the risk-return trade-off

### QUBO Formulation

For binary decisions, this becomes a **Quadratic Unconstrained Binary Optimization (QUBO)** problem:

$$\min_{\mathbf{x} \in \{0,1\}^n} \quad \mathbf{x}^T Q \mathbf{x}$$

where $Q_{ij} = \frac{1}{2}\Sigma_{ij} - q\,\mu_i \delta_{ij}$ encodes both the risk and return objectives.

### Ising Mapping

The QUBO maps to an Ising Hamiltonian via the substitution $x_i = \frac{1 - Z_i}{2}$:

$$H = \sum_{i<j} J_{ij} Z_i Z_j + \sum_i h_i Z_i + \text{const}$$

where:
- $J_{ij} = \frac{1}{4} Q_{ij}$ are the coupling coefficients
- $h_i = -\frac{1}{2}\left(Q_{ii} + \sum_{j \neq i} Q_{ij}\right)$ are the local fields

The ground state of this Hamiltonian corresponds to the optimal portfolio selection.

In [ ]:
# --- Setup 4-Stock Portfolio ---

# Asset names and parameters
assets = ["Tech (AAPL)", "Finance (JPM)", "Energy (XOM)", "Healthcare (JNJ)"]
n_assets = len(assets)

# Expected annual returns (as fractions)
mu = np.array([0.15, 0.10, 0.08, 0.12])

# Covariance matrix (annualized)
# Constructed to be realistic: tech is volatile, healthcare is defensive
sigma = np.array([
    [0.040, 0.012, 0.005, 0.008],   # Tech: high variance
    [0.012, 0.025, 0.010, 0.006],   # Finance: moderate
    [0.005, 0.010, 0.030, 0.004],   # Energy: moderate-high
    [0.008, 0.006, 0.004, 0.015],   # Healthcare: low variance
])

# Risk aversion parameter
q = 0.5

# Display portfolio parameters
print("Portfolio Setup")
print("=" * 50)
print(f"{'Asset':<20} {'Return':>8} {'Volatility':>12}")
print("-" * 50)
for i, name in enumerate(assets):
    print(f"{name:<20} {mu[i]:>7.1%} {np.sqrt(sigma[i,i]):>11.1%}")
print(f"\nRisk aversion (q): {q}")

# --- Visualize Correlation Matrix ---
# Compute correlation from covariance
std_devs = np.sqrt(np.diag(sigma))
corr = sigma / np.outer(std_devs, std_devs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Covariance heatmap
im1 = ax1.imshow(sigma, cmap="YlOrRd", aspect="auto")
ax1.set_xticks(range(n_assets))
ax1.set_yticks(range(n_assets))
ax1.set_xticklabels([a.split(" ")[0] for a in assets], rotation=45, ha="right")
ax1.set_yticklabels([a.split(" ")[0] for a in assets])
for i in range(n_assets):
    for j in range(n_assets):
        ax1.text(j, i, f"{sigma[i,j]:.4f}", ha="center", va="center", fontsize=10,
                color="white" if sigma[i,j] > 0.02 else "black")
ax1.set_title("Covariance Matrix $\\Sigma$", fontsize=13, fontweight="bold")
plt.colorbar(im1, ax=ax1, shrink=0.8)

# Correlation heatmap
im2 = ax2.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax2.set_xticks(range(n_assets))
ax2.set_yticks(range(n_assets))
ax2.set_xticklabels([a.split(" ")[0] for a in assets], rotation=45, ha="right")
ax2.set_yticklabels([a.split(" ")[0] for a in assets])
for i in range(n_assets):
    for j in range(n_assets):
        ax2.text(j, i, f"{corr[i,j]:.2f}", ha="center", va="center", fontsize=10,
                fontweight="bold")
ax2.set_title("Correlation Matrix", fontsize=13, fontweight="bold")
plt.colorbar(im2, ax=ax2, shrink=0.8)

fig.suptitle("4-Stock Portfolio: Risk Structure", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## Classical Solution

Before applying quantum methods, we solve the portfolio optimization problem classically using `scipy.optimize`. This provides the optimal benchmark against which to compare quantum results.

We solve two problems:
1. **Continuous relaxation**: Allow fractional weights $w_i \in [0, 1]$ (standard Markowitz)
2. **Binary (discrete)**: Exhaustive search over all $2^n$ binary portfolios to find the true optimum

In [ ]:
# --- Classical Solution ---

def portfolio_objective(w, sigma, mu, q):
    """Mean-variance objective: risk - q * return."""
    return 0.5 * w @ sigma @ w - q * mu @ w


def portfolio_return(w, mu):
    return mu @ w


def portfolio_risk(w, sigma):
    return np.sqrt(w @ sigma @ w)


# --- Continuous optimization (Markowitz) ---
from scipy.optimize import minimize

constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
bounds = [(0, 1) for _ in range(n_assets)]
w0 = np.ones(n_assets) / n_assets

result_cont = minimize(
    portfolio_objective, w0, args=(sigma, mu, q),
    method="SLSQP", bounds=bounds, constraints=constraints
)
w_opt_cont = result_cont.x

print("Classical Continuous Solution (Markowitz)")
print("=" * 50)
for i, name in enumerate(assets):
    print(f"  {name:<20}: {w_opt_cont[i]:>6.1%}")
print(f"  Expected Return      : {portfolio_return(w_opt_cont, mu):>6.2%}")
print(f"  Portfolio Risk       : {portfolio_risk(w_opt_cont, sigma):>6.2%}")
print(f"  Objective Value      : {result_cont.fun:>8.6f}")

# --- Binary (discrete) exhaustive search ---
best_binary_obj = np.inf
best_binary_w = None
all_binary_portfolios = []

for i in range(1, 2**n_assets):  # Skip empty portfolio
    bits = np.array([int(b) for b in format(i, f"0{n_assets}b")], dtype=float)
    # Normalize to sum to 1
    w = bits / bits.sum()
    obj = portfolio_objective(w, sigma, mu, q)
    ret = portfolio_return(w, mu)
    risk = portfolio_risk(w, sigma)
    all_binary_portfolios.append({
        "bits": bits, "weights": w, "objective": obj,
        "return": ret, "risk": risk, "label": format(i, f"0{n_assets}b")
    })
    if obj < best_binary_obj:
        best_binary_obj = obj
        best_binary_w = w
        best_binary_bits = bits

print(f"\nClassical Binary Solution (Exhaustive Search over {2**n_assets - 1} portfolios)")
print("=" * 50)
print(f"  Selected assets: {best_binary_bits}")
for i, name in enumerate(assets):
    if best_binary_bits[i] > 0:
        print(f"  {name:<20}: {best_binary_w[i]:>6.1%}")
print(f"  Expected Return      : {portfolio_return(best_binary_w, mu):>6.2%}")
print(f"  Portfolio Risk       : {portfolio_risk(best_binary_w, sigma):>6.2%}")
print(f"  Objective Value      : {best_binary_obj:>8.6f}")

# --- Efficient Frontier ---
n_frontier = 100
target_returns = np.linspace(min(mu), max(mu), n_frontier)
frontier_risks = []
frontier_returns = []

for target in target_returns:
    cons = [
        {"type": "eq", "fun": lambda w: np.sum(w) - 1},
        {"type": "eq", "fun": lambda w, t=target: mu @ w - t},
    ]
    res = minimize(
        lambda w: 0.5 * w @ sigma @ w, w0,
        method="SLSQP", bounds=bounds, constraints=cons
    )
    if res.success:
        frontier_risks.append(np.sqrt(res.x @ sigma @ res.x))
        frontier_returns.append(target)

# Plot efficient frontier
fig, ax = plt.subplots(figsize=(10, 7))

# Efficient frontier
ax.plot(frontier_risks, frontier_returns, "-", color="#1565C0", linewidth=2.5,
        label="Efficient Frontier")

# Individual assets
asset_colors = ["#E53935", "#43A047", "#FB8C00", "#8E24AA"]
for i, name in enumerate(assets):
    ax.scatter(np.sqrt(sigma[i,i]), mu[i], s=150, c=asset_colors[i],
              edgecolors="black", linewidths=1.5, zorder=5)
    ax.annotate(name, (np.sqrt(sigma[i,i]), mu[i]),
               xytext=(10, 5), textcoords="offset points", fontsize=10)

# Binary portfolios
for pf in all_binary_portfolios:
    ax.scatter(pf["risk"], pf["return"], s=40, c="gray", alpha=0.4, zorder=3)

# Optimal portfolios
ax.scatter(portfolio_risk(w_opt_cont, sigma), portfolio_return(w_opt_cont, mu),
          s=200, marker="*", c="gold", edgecolors="black", linewidths=1.5,
          zorder=6, label=f"Markowitz Optimal (q={q})")
ax.scatter(portfolio_risk(best_binary_w, sigma), portfolio_return(best_binary_w, mu),
          s=200, marker="D", c="#00BCD4", edgecolors="black", linewidths=1.5,
          zorder=6, label="Best Binary Portfolio")

ax.set_xlabel("Portfolio Risk (Std Dev)", fontsize=12)
ax.set_ylabel("Expected Return", fontsize=12)
ax.set_title("Efficient Frontier and Portfolio Candidates", fontsize=14, fontweight="bold")
ax.legend(fontsize=10, loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## QUBO Formulation

We now convert the portfolio optimization problem into a form suitable for quantum computation.

### Steps:
1. Build the **QUBO matrix** $Q$ from the covariance matrix $\Sigma$ and return vector $\mu$
2. Map to an **Ising Hamiltonian** using the substitution $x_i = \frac{1 - Z_i}{2}$
3. Express as a PennyLane Hamiltonian with Pauli-Z operators

In [ ]:
# --- QUBO and Ising Hamiltonian Construction ---

def build_qubo_matrix(sigma, mu, q, penalty=1.0, n_select=None):
    """
    Build QUBO matrix for portfolio optimization.
    
    Q_ij = (1/2) * Sigma_ij  - q * mu_i * delta_ij
    
    If n_select is given, add penalty: penalty * (sum(x) - n_select)^2
    """
    n = len(mu)
    Q = 0.5 * sigma.copy() - q * np.diag(mu)
    
    if n_select is not None:
        # Penalty for cardinality constraint: penalty * (sum_i x_i - n_select)^2
        # Expands to: penalty * (sum_i x_i^2 + 2 * sum_{i<j} x_i*x_j - 2*n_select*sum_i x_i + n_select^2)
        # Since x_i^2 = x_i for binary:
        for i in range(n):
            Q[i, i] += penalty * (1 - 2 * n_select)
        for i in range(n):
            for j in range(i+1, n):
                Q[i, j] += penalty
                Q[j, i] += penalty
    
    return Q


def qubo_to_ising(Q):
    """
    Convert QUBO matrix to Ising Hamiltonian coefficients.
    
    Using x_i = (1 - Z_i) / 2:
    H = sum_{i<j} J_ij Z_i Z_j + sum_i h_i Z_i + offset
    """
    n = Q.shape[0]
    
    # Make Q symmetric
    Q_sym = (Q + Q.T) / 2
    
    J = np.zeros((n, n))
    h = np.zeros(n)
    offset = 0.0
    
    for i in range(n):
        for j in range(i+1, n):
            J[i, j] = Q_sym[i, j] / 4
    
    for i in range(n):
        h[i] = -Q_sym[i, i] / 2
        for j in range(n):
            if i != j:
                h[i] -= Q_sym[i, j] / 4
    
    # Offset
    offset = np.sum(Q_sym) / 4 + np.trace(Q_sym) / 4
    
    return J, h, offset


def build_pennylane_hamiltonian(J, h, offset, n_qubits):
    """
    Build a PennyLane Hamiltonian from Ising coefficients.
    
    H = sum_{i<j} J_ij Z_i Z_j + sum_i h_i Z_i + offset * I
    """
    coeffs = []
    ops = []
    
    # ZZ terms
    for i in range(n_qubits):
        for j in range(i+1, n_qubits):
            if abs(J[i, j]) > 1e-10:
                coeffs.append(J[i, j])
                ops.append(qml.PauliZ(i) @ qml.PauliZ(j))
    
    # Z terms
    for i in range(n_qubits):
        if abs(h[i]) > 1e-10:
            coeffs.append(h[i])
            ops.append(qml.PauliZ(i))
    
    # Identity (offset)
    coeffs.append(offset)
    ops.append(qml.Identity(0))
    
    return qml.Hamiltonian(coeffs, ops)


# Build QUBO matrix
Q = build_qubo_matrix(sigma, mu, q)
print("QUBO Matrix Q:")
print(np.array2string(Q, precision=4, suppress_small=True))

# Convert to Ising
J, h, offset = qubo_to_ising(Q)
print(f"\nIsing Coupling Coefficients J_ij:")
print(np.array2string(J, precision=4, suppress_small=True))
print(f"\nIsing Local Fields h_i:")
print(np.array2string(h, precision=4, suppress_small=True))
print(f"\nIsing Offset: {offset:.6f}")

# Build PennyLane Hamiltonian
cost_hamiltonian = build_pennylane_hamiltonian(J, h, offset, n_assets)
print(f"\nPennyLane Cost Hamiltonian:")
print(cost_hamiltonian)

## QAOA Implementation

We now implement QAOA to find the ground state of the cost Hamiltonian.

### QAOA Circuit Structure

For $p$ QAOA layers, the circuit applies:

$$|\boldsymbol{\gamma}, \boldsymbol{\beta}\rangle = \prod_{l=1}^{p} e^{-i\beta_l H_M} e^{-i\gamma_l H_C} |+\rangle^{\otimes n}$$

where:
- $H_C$ is the cost Hamiltonian (encodes the optimization problem)
- $H_M = \sum_i X_i$ is the mixer Hamiltonian
- $\gamma_l, \beta_l$ are variational parameters optimized classically

In [ ]:
# --- QAOA Implementation ---

n_qubits_qaoa = n_assets
p_layers = 3  # QAOA depth

dev_qaoa = qml.device("default.qubit", wires=n_qubits_qaoa)


def qaoa_cost_layer(gamma, J, h, n_qubits):
    """Apply the cost unitary exp(-i * gamma * H_C)."""
    # ZZ interactions
    for i in range(n_qubits):
        for j in range(i+1, n_qubits):
            if abs(J[i, j]) > 1e-10:
                qml.CNOT(wires=[i, j])
                qml.RZ(2 * gamma * J[i, j], wires=j)
                qml.CNOT(wires=[i, j])
    
    # Z terms (local fields)
    for i in range(n_qubits):
        if abs(h[i]) > 1e-10:
            qml.RZ(2 * gamma * h[i], wires=i)


def qaoa_mixer_layer(beta, n_qubits):
    """Apply the mixer unitary exp(-i * beta * H_M) where H_M = sum_i X_i."""
    for i in range(n_qubits):
        qml.RX(2 * beta, wires=i)


@qml.qnode(dev_qaoa, interface="autograd")
def qaoa_circuit(params):
    """
    QAOA circuit for portfolio optimization.
    
    Args:
        params: Array of shape (2*p,) containing [gamma_1,...,gamma_p, beta_1,...,beta_p]
    
    Returns:
        Expectation value of the cost Hamiltonian
    """
    gammas = params[:p_layers]
    betas = params[p_layers:]
    
    # Initial superposition
    for i in range(n_qubits_qaoa):
        qml.Hadamard(wires=i)
    
    # QAOA layers
    for l in range(p_layers):
        qaoa_cost_layer(gammas[l], J, h, n_qubits_qaoa)
        qaoa_mixer_layer(betas[l], n_qubits_qaoa)
    
    return qml.expval(cost_hamiltonian)


@qml.qnode(dev_qaoa, interface="autograd")
def qaoa_probs(params):
    """Get output probabilities from the QAOA circuit."""
    gammas = params[:p_layers]
    betas = params[p_layers:]
    
    for i in range(n_qubits_qaoa):
        qml.Hadamard(wires=i)
    
    for l in range(p_layers):
        qaoa_cost_layer(gammas[l], J, h, n_qubits_qaoa)
        qaoa_mixer_layer(betas[l], n_qubits_qaoa)
    
    return qml.probs(wires=range(n_qubits_qaoa))


# --- Optimize QAOA ---
print("Optimizing QAOA Circuit")
print("=" * 50)
print(f"Number of qubits : {n_qubits_qaoa}")
print(f"QAOA depth (p)   : {p_layers}")
print(f"Parameters       : {2 * p_layers}")
print()

# Initialize parameters
init_params = pnp.array(
    np.random.uniform(0, np.pi, size=2 * p_layers),
    requires_grad=True
)

# PennyLane optimizer
opt = qml.GradientDescentOptimizer(stepsize=0.1)

n_steps = 100
qaoa_costs = []
params = init_params.copy()

for step in range(n_steps):
    params = opt.step(qaoa_circuit, params)
    cost = qaoa_circuit(params)
    qaoa_costs.append(float(cost))
    
    if (step + 1) % 20 == 0:
        print(f"Step {step+1:3d}/{n_steps}: Cost = {float(cost):.6f}")

print(f"\nFinal QAOA cost: {qaoa_costs[-1]:.6f}")

# --- Convergence Plot ---
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(1, n_steps+1), qaoa_costs, "-", color="#D32F2F", linewidth=2, label="QAOA Cost")
ax.axhline(y=best_binary_obj + offset, color="#1565C0", linestyle="--", linewidth=1.5,
           label=f"Optimal (Ising) = {best_binary_obj + offset:.4f}")
ax.set_xlabel("Optimization Step", fontsize=12)
ax.set_ylabel("Cost (Hamiltonian Expectation)", fontsize=12)
ax.set_title("QAOA Convergence", fontsize=14, fontweight="bold")
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# --- Extract Optimal Portfolio from QAOA ---

# Get probability distribution over computational basis states
probs = qaoa_probs(params)
probs_np = np.array(probs)

# Find the most probable state
max_idx = np.argmax(probs_np)
qaoa_bitstring = format(max_idx, f"0{n_assets}b")
qaoa_selection = np.array([int(b) for b in qaoa_bitstring], dtype=float)

# Compute portfolio metrics
if qaoa_selection.sum() > 0:
    qaoa_weights = qaoa_selection / qaoa_selection.sum()
else:
    qaoa_weights = np.ones(n_assets) / n_assets

qaoa_return = portfolio_return(qaoa_weights, mu)
qaoa_risk = portfolio_risk(qaoa_weights, sigma)
qaoa_obj = portfolio_objective(qaoa_weights, sigma, mu, q)

print("QAOA Optimal Portfolio")
print("=" * 50)
print(f"Most probable bitstring: {qaoa_bitstring} (prob = {probs_np[max_idx]:.4f})")
print(f"Selected assets:")
for i, name in enumerate(assets):
    status = "SELECTED" if qaoa_selection[i] > 0 else "---"
    weight = f"{qaoa_weights[i]:.1%}" if qaoa_selection[i] > 0 else "---"
    print(f"  {name:<20}: {status:<10} Weight: {weight}")
print(f"\nExpected Return: {qaoa_return:.2%}")
print(f"Portfolio Risk : {qaoa_risk:.2%}")
print(f"Objective Value: {qaoa_obj:.6f}")

# Compare with classical
print(f"\n--- Comparison with Classical Binary Optimum ---")
print(f"{'Metric':<25} {'QAOA':>12} {'Classical':>12} {'Gap':>10}")
print("-" * 60)
print(f"{'Objective':.<25} {qaoa_obj:>12.6f} {best_binary_obj:>12.6f} {abs(qaoa_obj - best_binary_obj):>10.6f}")
print(f"{'Return':.<25} {qaoa_return:>11.2%} {portfolio_return(best_binary_w, mu):>11.2%}")
print(f"{'Risk':.<25} {qaoa_risk:>11.2%} {portfolio_risk(best_binary_w, sigma):>11.2%}")

# Probability distribution bar chart
fig, ax = plt.subplots(figsize=(12, 5))
n_states = 2**n_assets
x_labels = [format(i, f"0{n_assets}b") for i in range(n_states)]
colors = ["#D32F2F" if i == max_idx else "#90CAF9" for i in range(n_states)]

# Highlight the classical optimum
classical_idx = int("".join([str(int(b)) for b in best_binary_bits]), 2)
colors[classical_idx] = "#1565C0"

ax.bar(range(n_states), probs_np, color=colors, edgecolor="black", linewidth=0.5)
ax.set_xticks(range(n_states))
ax.set_xticklabels(x_labels, rotation=45, ha="right", fontsize=9)
ax.set_xlabel("Bitstring (Asset Selection)", fontsize=12)
ax.set_ylabel("Probability", fontsize=12)
ax.set_title("QAOA Output Probabilities", fontsize=14, fontweight="bold")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#D32F2F", edgecolor="black", label=f"QAOA Best: {qaoa_bitstring}"),
    Patch(facecolor="#1565C0", edgecolor="black",
          label=f"Classical Opt: {''.join([str(int(b)) for b in best_binary_bits])}"),
]
ax.legend(handles=legend_elements, fontsize=10)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

## VQE Approach

As an alternative to QAOA, we implement the **Variational Quantum Eigensolver (VQE)** with a hardware-efficient ansatz. While QAOA uses a problem-specific circuit structure, VQE employs a general-purpose parameterized circuit that may converge faster for small problem sizes.

### Hardware-Efficient Ansatz

The ansatz consists of:
- Single-qubit rotations $R_Y(\theta)$ and $R_Z(\phi)$ on each qubit
- Entangling CNOT gates in a linear chain
- Multiple repetitions (layers) of this pattern

In [ ]:
# --- VQE with Hardware-Efficient Ansatz ---

n_vqe_layers = 3
dev_vqe = qml.device("default.qubit", wires=n_assets)


@qml.qnode(dev_vqe, interface="autograd")
def vqe_circuit(params):
    """
    VQE circuit with hardware-efficient ansatz.
    
    Args:
        params: shape (n_vqe_layers, n_assets, 2) for RY and RZ rotations
    """
    params_reshaped = params.reshape(n_vqe_layers, n_assets, 2)
    
    for layer in range(n_vqe_layers):
        # Single-qubit rotations
        for i in range(n_assets):
            qml.RY(params_reshaped[layer, i, 0], wires=i)
            qml.RZ(params_reshaped[layer, i, 1], wires=i)
        
        # Entangling layer
        for i in range(n_assets - 1):
            qml.CNOT(wires=[i, i + 1])
        # Circular entanglement
        if n_assets > 2:
            qml.CNOT(wires=[n_assets - 1, 0])
    
    return qml.expval(cost_hamiltonian)


@qml.qnode(dev_vqe, interface="autograd")
def vqe_probs(params):
    """Get output probabilities from VQE circuit."""
    params_reshaped = params.reshape(n_vqe_layers, n_assets, 2)
    
    for layer in range(n_vqe_layers):
        for i in range(n_assets):
            qml.RY(params_reshaped[layer, i, 0], wires=i)
            qml.RZ(params_reshaped[layer, i, 1], wires=i)
        for i in range(n_assets - 1):
            qml.CNOT(wires=[i, i + 1])
        if n_assets > 2:
            qml.CNOT(wires=[n_assets - 1, 0])
    
    return qml.probs(wires=range(n_assets))


# --- Optimize VQE ---
print("Optimizing VQE Circuit")
print("=" * 50)
n_vqe_params = n_vqe_layers * n_assets * 2
print(f"Number of qubits     : {n_assets}")
print(f"Ansatz layers        : {n_vqe_layers}")
print(f"Trainable parameters : {n_vqe_params}")
print()

vqe_init_params = pnp.array(
    np.random.uniform(0, 2 * np.pi, size=n_vqe_params),
    requires_grad=True
)

vqe_opt = qml.AdamOptimizer(stepsize=0.05)
vqe_params = vqe_init_params.copy()
vqe_costs = []

n_vqe_steps = 150

for step in range(n_vqe_steps):
    vqe_params = vqe_opt.step(vqe_circuit, vqe_params)
    cost = vqe_circuit(vqe_params)
    vqe_costs.append(float(cost))
    
    if (step + 1) % 30 == 0:
        print(f"Step {step+1:3d}/{n_vqe_steps}: Cost = {float(cost):.6f}")

print(f"\nFinal VQE cost: {vqe_costs[-1]:.6f}")

# Extract VQE result
vqe_probs_out = np.array(vqe_probs(vqe_params))
vqe_max_idx = np.argmax(vqe_probs_out)
vqe_bitstring = format(vqe_max_idx, f"0{n_assets}b")
vqe_selection = np.array([int(b) for b in vqe_bitstring], dtype=float)

if vqe_selection.sum() > 0:
    vqe_weights = vqe_selection / vqe_selection.sum()
else:
    vqe_weights = np.ones(n_assets) / n_assets

print(f"\nVQE Result:")
print(f"  Bitstring  : {vqe_bitstring} (prob = {vqe_probs_out[vqe_max_idx]:.4f})")
print(f"  Return     : {portfolio_return(vqe_weights, mu):.2%}")
print(f"  Risk       : {portfolio_risk(vqe_weights, sigma):.2%}")
print(f"  Objective  : {portfolio_objective(vqe_weights, sigma, mu, q):.6f}")

In [ ]:
# --- Comprehensive Comparison ---

fig = plt.figure(figsize=(18, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# --- Panel 1: Convergence comparison ---
ax1 = fig.add_subplot(gs[0, 0])
ax1.plot(range(1, n_steps+1), qaoa_costs, "-", color="#D32F2F", linewidth=2, label="QAOA", alpha=0.9)
ax1.plot(range(1, n_vqe_steps+1), vqe_costs, "-", color="#1565C0", linewidth=2, label="VQE", alpha=0.9)
ax1.set_xlabel("Optimization Step")
ax1.set_ylabel("Cost")
ax1.set_title("Convergence Comparison", fontweight="bold")
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Panel 2: Portfolio Allocations Bar Chart ---
ax2 = fig.add_subplot(gs[0, 1])
x = np.arange(n_assets)
width = 0.2

bars1 = ax2.bar(x - 1.5*width, w_opt_cont, width, label="Markowitz", color="#4CAF50", edgecolor="black", linewidth=0.5)
bars2 = ax2.bar(x - 0.5*width, best_binary_w, width, label="Classical Binary", color="#FF9800", edgecolor="black", linewidth=0.5)
bars3 = ax2.bar(x + 0.5*width, qaoa_weights, width, label="QAOA", color="#D32F2F", edgecolor="black", linewidth=0.5)
bars4 = ax2.bar(x + 1.5*width, vqe_weights, width, label="VQE", color="#1565C0", edgecolor="black", linewidth=0.5)

ax2.set_xticks(x)
ax2.set_xticklabels([a.split(" ")[0] for a in assets], fontsize=10)
ax2.set_ylabel("Weight")
ax2.set_title("Portfolio Allocations", fontweight="bold")
ax2.legend(fontsize=9, loc="upper right")
ax2.grid(True, alpha=0.3, axis="y")

# --- Panel 3: Risk-Return Plot ---
ax3 = fig.add_subplot(gs[0, 2])

# Efficient frontier
ax3.plot(frontier_risks, frontier_returns, "-", color="gray", linewidth=1.5, alpha=0.5, label="Efficient Frontier")

# Individual assets
for i, name in enumerate(assets):
    ax3.scatter(np.sqrt(sigma[i,i]), mu[i], s=80, c=asset_colors[i], edgecolors="black",
              linewidths=1, zorder=4, alpha=0.6)

# Portfolio solutions
methods = [
    ("Markowitz", w_opt_cont, "*", "#4CAF50", 250),
    ("Classical Binary", best_binary_w, "D", "#FF9800", 150),
    ("QAOA", qaoa_weights, "^", "#D32F2F", 150),
    ("VQE", vqe_weights, "s", "#1565C0", 150),
]
for name, w, marker, color, size in methods:
    r = portfolio_risk(w, sigma)
    ret = portfolio_return(w, mu)
    ax3.scatter(r, ret, s=size, marker=marker, c=color, edgecolors="black",
              linewidths=1.5, zorder=5, label=name)

ax3.set_xlabel("Risk (Std Dev)")
ax3.set_ylabel("Expected Return")
ax3.set_title("Risk-Return Comparison", fontweight="bold")
ax3.legend(fontsize=8, loc="lower right")
ax3.grid(True, alpha=0.3)

# --- Panel 4: Objective Values Bar Chart ---
ax4 = fig.add_subplot(gs[1, 0])
method_names = ["Markowitz", "Classical\nBinary", "QAOA", "VQE"]
obj_values = [
    portfolio_objective(w_opt_cont, sigma, mu, q),
    best_binary_obj,
    portfolio_objective(qaoa_weights, sigma, mu, q),
    portfolio_objective(vqe_weights, sigma, mu, q),
]
bar_colors = ["#4CAF50", "#FF9800", "#D32F2F", "#1565C0"]
bars = ax4.bar(method_names, obj_values, color=bar_colors, edgecolor="black", linewidth=0.8)
for bar, val in zip(bars, obj_values):
    ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.0005,
             f"{val:.4f}", ha="center", fontsize=10, fontweight="bold")
ax4.set_ylabel("Objective Value (lower = better)")
ax4.set_title("Objective Value Comparison", fontweight="bold")
ax4.grid(True, alpha=0.3, axis="y")

# --- Panel 5: QAOA Probability Distribution ---
ax5 = fig.add_subplot(gs[1, 1])
top_k = 6
sorted_indices = np.argsort(probs_np)[::-1][:top_k]
top_labels = [format(i, f"0{n_assets}b") for i in sorted_indices]
top_probs = probs_np[sorted_indices]
ax5.barh(range(top_k), top_probs, color="#D32F2F", edgecolor="black", linewidth=0.5)
ax5.set_yticks(range(top_k))
ax5.set_yticklabels(top_labels, fontsize=10, fontfamily="monospace")
ax5.set_xlabel("Probability")
ax5.set_title("QAOA: Top Bitstrings", fontweight="bold")
ax5.invert_yaxis()
ax5.grid(True, alpha=0.3, axis="x")

# --- Panel 6: Optimality Gap ---
ax6 = fig.add_subplot(gs[1, 2])
optimal_obj = min(obj_values)
gaps = [(v - optimal_obj) / abs(optimal_obj) * 100 if abs(optimal_obj) > 1e-10 else 0
        for v in obj_values]
ax6.bar(method_names, gaps, color=bar_colors, edgecolor="black", linewidth=0.8)
for i, (bar_pos, gap) in enumerate(zip(method_names, gaps)):
    ax6.text(i, gaps[i] + 0.2, f"{gap:.1f}%", ha="center", fontsize=10, fontweight="bold")
ax6.set_ylabel("Optimality Gap (%)")
ax6.set_title("Optimality Gap vs Best Solution", fontweight="bold")
ax6.grid(True, alpha=0.3, axis="y")

fig.suptitle("Quantum Portfolio Optimization: Comprehensive Comparison",
             fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Scaling Analysis

A critical question for quantum portfolio optimization is: **how do resource requirements scale with the number of assets?**

Key scaling considerations:
- **Qubits**: $n$ qubits for $n$ assets (binary encoding) or $\lceil \log_2 K \rceil \cdot n$ for $K$-level discretization
- **Circuit depth**: Depends on the number of QAOA layers $p$ and the density of the Ising coupling graph
- **Classical optimization**: The number of variational parameters grows linearly with $p$ but the optimization landscape becomes more complex

In [ ]:
# --- Scaling Analysis ---

asset_counts = [4, 6, 8, 10, 12, 16, 20, 30, 50]
qaoa_depths = [1, 2, 3, 5]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# --- Panel 1: Circuit Depth vs Assets ---
ax = axes[0]
for p in qaoa_depths:
    # Each QAOA layer has: n*(n-1)/2 ZZ gates (2 CNOTs + 1 RZ each) + n RZ + n RX
    # Approximate depth: p * (3 * n*(n-1)/2 + 2*n)
    depths = [p * (3 * n*(n-1)//2 + 2*n) for n in asset_counts]
    ax.plot(asset_counts, depths, "o-", linewidth=2, markersize=6, label=f"p = {p}")

ax.set_xlabel("Number of Assets", fontsize=12)
ax.set_ylabel("Approximate Circuit Depth", fontsize=12)
ax.set_title("QAOA Circuit Depth Scaling", fontsize=13, fontweight="bold")
ax.legend(title="QAOA Layers", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_yscale("log")

# --- Panel 2: Number of Parameters ---
ax = axes[1]
for p in qaoa_depths:
    qaoa_params = [2 * p for _ in asset_counts]  # QAOA: 2p parameters (gamma + beta)
    vqe_params = [p * n * 2 for n in asset_counts]  # VQE: p * n * 2 parameters

ax.plot(asset_counts, [2*p for p in [1]*len(asset_counts)], "--", color="gray", alpha=0.5)

for p in qaoa_depths:
    vqe_p = [p * n * 2 for n in asset_counts]
    ax.plot(asset_counts, vqe_p, "s-", linewidth=2, markersize=5, label=f"VQE (L={p})")

# QAOA has very few params
for p in qaoa_depths:
    ax.axhline(y=2*p, linestyle=":", alpha=0.4)
    ax.text(asset_counts[-1]+1, 2*p, f"QAOA p={p}", fontsize=8, va="center")

ax.set_xlabel("Number of Assets", fontsize=12)
ax.set_ylabel("Number of Parameters", fontsize=12)
ax.set_title("Variational Parameters Scaling", fontsize=13, fontweight="bold")
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_yscale("log")

# --- Panel 3: Solution Space Size ---
ax = axes[2]
classical_space = [2**n for n in asset_counts]
ax.semilogy(asset_counts, classical_space, "o-", color="#D32F2F", linewidth=2.5,
           markersize=7, label="Classical Search Space ($2^n$)")
ax.fill_between(asset_counts, classical_space, alpha=0.1, color="#D32F2F")

# Quantum advantage region
ax.axhline(y=1e6, color="#4CAF50", linestyle="--", linewidth=1.5,
           label="Practical classical limit")

ax.set_xlabel("Number of Assets", fontsize=12)
ax.set_ylabel("Number of Portfolios", fontsize=12)
ax.set_title("Combinatorial Explosion", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

fig.suptitle("Scaling Analysis: Quantum Portfolio Optimization",
             fontsize=15, fontweight="bold", y=1.02)
plt.tight_layout()
plt.show()

# Print scaling table
print("\nScaling Summary")
print("=" * 65)
print(f"{'Assets':>8} {'Qubits':>8} {'Search Space':>14} {'QAOA(p=3) Depth':>17}")
print("-" * 65)
for n in asset_counts:
    depth = 3 * (3 * n*(n-1)//2 + 2*n)
    print(f"{n:>8} {n:>8} {2**n:>14,} {depth:>17,}")

## Conclusion

### Summary

In this project, we implemented and compared multiple approaches to portfolio optimization:

1. **Classical Markowitz** (continuous): The gold standard providing the globally optimal continuous solution
2. **Classical Binary Search**: Exhaustive enumeration of all $2^n$ binary portfolios
3. **QAOA**: Quantum Approximate Optimization Algorithm with $p=3$ layers
4. **VQE**: Variational Quantum Eigensolver with a hardware-efficient ansatz

### Key Findings

- **QUBO-to-Ising mapping** successfully encodes the mean-variance portfolio problem into a quantum Hamiltonian
- **QAOA** can find competitive solutions with very few variational parameters ($2p$), but requires many optimization steps
- **VQE** with a hardware-efficient ansatz has more parameters but can achieve better convergence in some cases
- The **optimality gap** between quantum and classical solutions decreases with more optimization steps and deeper circuits

### Scaling Considerations

- For $n$ assets, the classical brute-force search explores $2^n$ portfolios, which becomes intractable for $n > 30$
- Quantum approaches use $n$ qubits and polynomial-depth circuits, offering a potential exponential speedup
- However, current NISQ devices are limited to approximately 100 noisy qubits, restricting practical portfolio sizes

### Future Directions

- **Constrained QAOA**: Incorporate budget and cardinality constraints directly into the mixer Hamiltonian
- **Warm-starting**: Use classical solutions to initialize quantum parameters
- **Error mitigation**: Apply zero-noise extrapolation and probabilistic error cancellation for NISQ execution
- **Multi-period optimization**: Extend to dynamic portfolio rebalancing

### References

1. Markowitz, H. "Portfolio Selection." *The Journal of Finance* 7(1), 77-91 (1952).
2. Farhi, E., Goldstone, J. & Gutmann, S. "A Quantum Approximate Optimization Algorithm." *arXiv:1411.4028* (2014).
3. Brandhofer, S. et al. "Benchmarking the performance of portfolio optimization with QAOA." *Quantum Science and Technology* 8, 024002 (2023).
4. Slate, N. et al. "Quantum walk-based vehicle routing optimisation." *Frontiers in Physics* 9, 730856 (2021).
5. PennyLane QAOA Tutorial: https://pennylane.ai/qml/demos/tutorial_qaoa_intro.html
6. Peruzzo, A. et al. "A variational eigenvalue solver on a photonic quantum processor." *Nature Communications* 5, 4213 (2014).